# Knowledge Analysis

Where does a transformer store and recall factual knowledge?
This notebook explores tools for tracing knowledge through the model,
identifying responsible neurons, and understanding MLP vs attention contributions.

This notebook covers the `irtk.knowledge` module.

## Why This Matters

Knowledge neurons are MLP neurons that store specific factual associations. Locating where facts are stored (via causal tracing) and understanding how MLPs recall them is essential for fact editing, understanding hallucination, and measuring what a model actually knows vs. guesses.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Meng et al. (2022) "Locating and Editing Factual Associations"](https://arxiv.org/abs/2202.05262) — Causal tracing to locate factual knowledge

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import knowledge

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, d_mlp={model.cfg.d_mlp}")

## 1. Knowledge Neurons

Which MLP neurons are most responsible for predicting a specific token?
`knowledge_neurons` computes each neuron's contribution to the target logit
(activation * projection onto the unembedding direction).

In [ ]:
# "The Eiffel Tower is in" -> Paris
prompt = "The Eiffel Tower is in"
tokens = model.to_tokens(prompt)
target_token = tokenizer.encode(" Paris")[0]

# Verify the model predicts this
logits = model(tokens)
top_pred = int(jnp.argmax(logits[-1]))
print(f"Prompt: {prompt!r}")
print(f"Top prediction: {tokenizer.decode([top_pred])!r}")
print(f"Target token (' Paris'): {target_token}")

In [ ]:
# Find the most important neurons for predicting " Paris"
kn = knowledge.knowledge_neurons(model, tokens, target_token=target_token, top_k=20)

print("Top 20 knowledge neurons for ' Paris':")
for entry in kn:
    print(f"  L{entry['layer']} N{entry['neuron']:4d}: "
          f"attr={entry['attribution']:+.4f}  act={entry['activation']:.4f}")

In [ ]:
# Visualize knowledge neuron distribution across layers
layer_counts = np.zeros(model.cfg.n_layers)
for entry in kn:
    layer_counts[entry["layer"]] += 1

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(model.cfg.n_layers), layer_counts)
ax.set_xlabel("Layer")
ax.set_ylabel("Count in Top 20")
ax.set_title("Knowledge Neurons by Layer")
plt.tight_layout()
plt.show()

## 2. Causal Knowledge Tracing

Corrupt the subject tokens and measure how much restoring activations
at each layer recovers the correct prediction. This reveals where
factual knowledge is computed.

In [ ]:
# Identify subject positions ("Eiffel Tower")
token_strs = [tokenizer.decode([int(t)]) for t in tokens]
print(f"Tokens: {token_strs}")

# Positions of "Eiffel Tower" tokens (adjust based on tokenization)
subject_pos = [1, 2, 3]  # These may vary with tokenizer
print(f"Subject positions: {subject_pos}")
print(f"Subject tokens: {[token_strs[i] for i in subject_pos]}")

In [ ]:
# Run causal knowledge tracing
trace = knowledge.causal_knowledge_tracing(
    model, tokens, subject_pos=subject_pos,
    target_token=target_token, noise_std=3.0,
)

print(f"Clean logit: {trace['clean_logit']:.4f}")
print(f"Corrupted logit: {trace['corrupted_logit']:.4f}")
print(f"Recovery (corrupted -> clean): {trace['clean_logit'] - trace['corrupted_logit']:.4f}")

In [ ]:
# Plot restoration by layer and component type
fig, ax = plt.subplots(figsize=(12, 5))
layers = range(model.cfg.n_layers)

ax.plot(layers, trace["restored_resid"], "o-", label="Restored residual", linewidth=2)
ax.plot(layers, trace["restored_mlp"], "s-", label="Restored MLP", linewidth=2)
ax.plot(layers, trace["restored_attn"], "^-", label="Restored attention", linewidth=2)
ax.axhline(trace["clean_logit"], color="green", linestyle="--", alpha=0.7, label="Clean")
ax.axhline(trace["corrupted_logit"], color="red", linestyle="--", alpha=0.7, label="Corrupted")

ax.set_xlabel("Layer")
ax.set_ylabel(f"Logit for ' Paris'")
ax.set_title("Causal Knowledge Tracing")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Fact Editing Vector

Compute a direction vector that, when added to the residual stream,
shifts the prediction from one token to another.

In [ ]:
# Edit "Paris" -> "London"
old_token = tokenizer.encode(" Paris")[0]
new_token = tokenizer.encode(" London")[0]

edit_vec = knowledge.fact_editing_vector(
    model, tokens, old_token=old_token, new_token=new_token, layer=8
)
print(f"Edit vector shape: {edit_vec.shape}")
print(f"Edit vector norm: {np.linalg.norm(edit_vec):.4f}")

# Apply the edit via steering
from irtk.steering import add_vector

for alpha in [0, 5, 10, 20, 50]:
    edited_logits = add_vector(
        model, tokens, "blocks.8.hook_resid_post",
        jnp.array(edit_vec), alpha=alpha,
    )
    top_pred = int(jnp.argmax(edited_logits[-1]))
    paris_logit = float(edited_logits[-1, old_token])
    london_logit = float(edited_logits[-1, new_token])
    print(f"  alpha={alpha:3d}: top={tokenizer.decode([top_pred]):>10s}  "
          f"Paris={paris_logit:+.2f}  London={london_logit:+.2f}")

## 4. MLP vs Attention Attribution

Break down the prediction into MLP and attention contributions at each layer.

In [ ]:
attr = knowledge.attribute_to_mlp_vs_attn(model, tokens, target_token=target_token)

print(f"Embedding contribution: {attr['embed_contrib']:.4f}")
print(f"Total logit: {attr['total_logit']:.4f}")
print(f"\nPer-layer breakdown:")
for l in range(model.cfg.n_layers):
    print(f"  L{l:2d}: MLP={attr['mlp_contrib'][l]:+.4f}  Attn={attr['attn_contrib'][l]:+.4f}")

In [ ]:
# Stacked bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = range(model.cfg.n_layers)
width = 0.35

ax.bar([i - width/2 for i in x], attr["mlp_contrib"], width, label="MLP", color="steelblue")
ax.bar([i + width/2 for i in x], attr["attn_contrib"], width, label="Attention", color="coral")
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel("Layer")
ax.set_ylabel(f"Contribution to ' Paris' logit")
ax.set_title("MLP vs Attention Contributions by Layer")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cumulative contributions
cum_mlp = np.cumsum(attr["mlp_contrib"])
cum_attn = np.cumsum(attr["attn_contrib"])
cum_total = attr["embed_contrib"] + cum_mlp + cum_attn

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, cum_mlp, "o-", label="Cumulative MLP")
ax.plot(x, cum_attn, "s-", label="Cumulative Attention")
ax.plot(x, cum_total, "^-", label="Cumulative Total")
ax.axhline(attr["total_logit"], color='gray', linestyle='--', alpha=0.5, label=f"Total logit ({attr['total_logit']:.2f})")
ax.set_xlabel("Layer")
ax.set_ylabel("Cumulative Contribution")
ax.set_title("Cumulative Knowledge Building")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `knowledge_neurons()` | Identify MLP neurons responsible for a prediction |
| `causal_knowledge_tracing()` | Trace where factual knowledge is computed via corruption/restoration |
| `fact_editing_vector()` | Compute a direction for editing factual predictions |
| `attribute_to_mlp_vs_attn()` | Break down predictions into MLP vs attention contributions |